# Model Training & Evaluation — Random Forest

Mục tiêu:
- Tái tạo pipeline tiền xử lý từ EDA (impute → encode → scale).
- Chia tập train/test (stratified 80/20).
- Huấn luyện Random Forest với tìm kiếm siêu tham số qua `GridSearchCV`.
- Đánh giá mô hình: Accuracy, Precision, Recall, F1, ROC-AUC.
- Lưu kết quả và biểu đồ vào `reports/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
PAIRED_COLORS = plt.get_cmap("Paired").colors

PROJECT_ROOT = Path("..").resolve()
DATA_PATH    = PROJECT_ROOT / "data" / "raw" / "heart.csv"
REPORTS_DIR  = PROJECT_ROOT / "reports"
FIGURES_DIR  = REPORTS_DIR / "figures"
TABLES_DIR   = REPORTS_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print("Setup complete.")

## 1. Load & Preprocess Data

Lặp lại pipeline tiền xử lý từ EDA: load → impute → binarize target → tách features/label.

In [ ]:
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope",
    "ca", "thal", "target",
]

df = pd.read_csv(DATA_PATH, names=columns, na_values="?")

# Impute missing values with mode (ca, thal are categorical)
for col in ["ca", "thal"]:
    df[col] = df[col].fillna(df[col].mode(dropna=True)[0])

# Binarize target: 0 = no disease, 1 = disease
df["target"] = (df["target"] > 0).astype(int)

# Cast categorical columns to int
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
numeric_features     = ["age", "trestbps", "chol", "thalach", "oldpeak"]
for col in categorical_features:
    df[col] = df[col].astype(int)

X = df[numeric_features + categorical_features]
y = df["target"]

print(f"X shape: {X.shape}  |  class balance: {y.value_counts().to_dict()}")
display(X.head())

## 2. Train / Test Split

Stratified split 80/20 để đảm bảo tỷ lệ nhãn được bảo toàn trong cả hai tập.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

split_summary = pd.DataFrame({
    "Split": ["Train", "Test"],
    "Size":  [len(X_train), len(X_test)],
    "Disease %": [
        round(y_train.mean() * 100, 2),
        round(y_test.mean()  * 100, 2),
    ],
})
display(split_summary)

## 3. Preprocessing Pipeline

- **Numeric**: `StandardScaler` — giúp ổn định khi so sánh feature importance và phù hợp nếu thêm mô hình khác sau này.
- **Categorical**: `OneHotEncoder` — theo khuyến nghị từ EDA cho `cp`, `restecg`, `slope`, `ca`, `thal`. Các biến binary (`sex`, `fbs`, `exang`) giữ nguyên (drop='if_binary').

In [ ]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    drop="if_binary",   # keep binary cols as 1 col, OHE the rest
    sparse_output=False,
    handle_unknown="ignore",
)

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,     numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# Fit on train, inspect output shape
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

# Recover feature names for later interpretation
ohe_names   = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
feature_names = numeric_features + list(ohe_names)

print(f"Processed feature count: {X_train_proc.shape[1]}")
print("Features:", feature_names)

## 4. Train Random Forest (GridSearchCV)

Dùng `StratifiedKFold` 5-fold CV để tìm siêu tham số tối ưu.
Metric chính là **ROC-AUC** vì đây là bài toán phân loại y tế (cần cân bằng recall và precision).

In [ ]:
param_grid = {
    "n_estimators":      [100, 200, 300],
    "max_depth":         [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf":  [1, 2],
    "max_features":      ["sqrt", "log2"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf = RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

grid_search.fit(X_train_proc, y_train)

best_model = grid_search.best_estimator_
print("\nBest params:", grid_search.best_params_)
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

In [ ]:
# Save top-10 CV results
cv_results = (
    pd.DataFrame(grid_search.cv_results_)
    [["params", "mean_test_score", "std_test_score", "rank_test_score"]]
    .sort_values("rank_test_score")
    .head(10)
    .reset_index(drop=True)
)
display(cv_results)
cv_results.to_csv(TABLES_DIR / "rf_cv_results_top10.csv", index=False)

## 5. Evaluate on Test Set

In [ ]:
y_pred      = best_model.predict(X_test_proc)
y_prob      = best_model.predict_proba(X_test_proc)[:, 1]

metrics = {
    "Accuracy":  round(accuracy_score(y_test, y_pred),  4),
    "Precision": round(precision_score(y_test, y_pred), 4),
    "Recall":    round(recall_score(y_test, y_pred),    4),
    "F1-Score":  round(f1_score(y_test, y_pred),        4),
    "ROC-AUC":   round(roc_auc_score(y_test, y_prob),   4),
}

metrics_df = pd.DataFrame(metrics.items(), columns=["Metric", "Value"])
display(metrics_df)
metrics_df.to_csv(TABLES_DIR / "rf_test_metrics.csv", index=False)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

### 5.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Disease", "Disease"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — Random Forest")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rf_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

### 5.2 ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score   = metrics["ROC-AUC"]

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color=PAIRED_COLORS[5], lw=2, label=f"ROC curve (AUC = {auc_score:.4f})")
ax.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Random Forest")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rf_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

### 5.3 Feature Importance

Dùng Mean Decrease in Impurity (MDI) của best model để xác định features quan trọng nhất.

In [ ]:
importances = best_model.feature_importances_
fi_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(fi_df.head(15))
fi_df.to_csv(TABLES_DIR / "rf_feature_importance.csv", index=False)

# Plot top-15
top15 = fi_df.head(15)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=top15, y="feature", x="importance",
    hue="feature", palette="Blues_r", legend=False, ax=ax,
)
ax.set_title("Top-15 Feature Importances — Random Forest")
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rf_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

### 5.4 Threshold Tuning (optional)

Trong y tế, thường ưu tiên Recall (giảm FN — bỏ sót bệnh nhân). Tìm ngưỡng tối ưu theo F1.

In [ ]:
thresholds  = np.arange(0.1, 0.9, 0.01)
thresh_rows = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    thresh_rows.append({
        "threshold": round(t, 2),
        "precision": precision_score(y_test, y_pred_t, zero_division=0),
        "recall":    recall_score(y_test, y_pred_t,    zero_division=0),
        "f1":        f1_score(y_test, y_pred_t,        zero_division=0),
    })

thresh_df     = pd.DataFrame(thresh_rows)
best_thresh   = thresh_df.loc[thresh_df["f1"].idxmax()]
print("Best threshold by F1:")
display(best_thresh.to_frame().T)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresh_df["threshold"], thresh_df["precision"], label="Precision", color=PAIRED_COLORS[1])
ax.plot(thresh_df["threshold"], thresh_df["recall"],    label="Recall",    color=PAIRED_COLORS[5])
ax.plot(thresh_df["threshold"], thresh_df["f1"],        label="F1-Score",  color=PAIRED_COLORS[3], lw=2)
ax.axvline(best_thresh["threshold"], color="gray", linestyle="--", label=f"Best threshold = {best_thresh['threshold']}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 vs. Decision Threshold")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "rf_threshold_tuning.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Summary

In [ ]:
summary_lines = [
    "# Model Training & Evaluation Summary — Random Forest",
    "",
    "## Best Hyperparameters (GridSearchCV, 5-fold CV, scoring=roc_auc)",
    "",
]
for k, v in grid_search.best_params_.items():
    summary_lines.append(f"- `{k}`: {v}")

summary_lines += [
    "",
    f"- Best CV ROC-AUC: **{grid_search.best_score_:.4f}**",
    "",
    "## Test-Set Metrics",
    "",
    metrics_df.to_markdown(index=False),
    "",
    "## Threshold Tuning",
    f"- Optimal decision threshold (max F1): **{best_thresh['threshold']}**",
    f"  - Precision: {best_thresh['precision']:.4f}",
    f"  - Recall:    {best_thresh['recall']:.4f}",
    f"  - F1:        {best_thresh['f1']:.4f}",
    "",
    "## Top-5 Features by Importance",
    "",
    fi_df.head(5).to_markdown(index=False),
]

summary_text = "\n".join(summary_lines) + "\n"
summary_path = REPORTS_DIR / "rf_model_summary.md"
summary_path.write_text(summary_text, encoding="utf-8")
print(f"Saved model summary to: {summary_path}")